# Комментарий

Скопируй тестовый набор данных

На отдельной вкладке внутри твоей копии собери ссылки на файлы для проверки заданий №2 и №3. Итого в твоей таблице должны быть следующие вкладки: 
1. Data
2. Сводная таблица из Data
3. Вкладка с ссылками на визуализацию и ссылкой на Jupyter Notebook

Не забудь расшерить доступ по ссылке с возможностью оставить комментарий 

In [1]:
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt

import scipy
from scipy import stats

for i in (np, pd, matplotlib, scipy):
    print(i.__version__)

2.3.3
2.3.3
3.10.8
1.16.3


In [2]:
df = pd.read_csv("./Data для тестового - Data.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Название рассылки       218 non-null    object
 1   Название кампании       218 non-null    object
 2   Направление             218 non-null    object
 3   Месяц                   218 non-null    object
 4   Дата                    218 non-null    object
 5   Год                     218 non-null    int64 
 6   Номер недели            218 non-null    int64 
 7   День недели             218 non-null    int64 
 8   День недели.1           218 non-null    object
 9   Время                   218 non-null    object
 10  Веб-версия              218 non-null    object
 11  Тема письма             218 non-null    object
 12  Сегмент                 218 non-null    object
 13  Отправлено              218 non-null    object
 14  Доставлено              218 non-null    object
 15  Открыт

In [3]:
def extract_digit(data):
    if data.dtype in ("int", "float"):
        return data
        
    return data.str.extract("([0-9]+)").astype("float64")[0]


cols2digit = ["Отправлено", "Доставлено", "Открытия", "Клики", "Отписки"] + [f"Воронка продаж. Шаг {i}" for i in range(1, 4)]
for title in cols2digit: 
    df[title] = extract_digit(df[title])

df["Дата"] = pd.to_datetime(df["Дата"])

C:\Users\Егор\AppData\Local\Temp\ipykernel_6728\2369784030.py:12: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Дата"] = pd.to_datetime(df["Дата"])


# Задание №1

Сделай сводную таблицу для набора данных из этого файла, которая отразит динамику основных метрик для email-рассылок (Delivery rate, Open rate, CTOR, UR).

In [4]:
pivot_table = df.groupby(["Дата"]).agg({title: "sum" for title in cols2digit})

def to_percent(data):
    return data.round(4)

pivot_table["Deliver rate"] = to_percent(pivot_table["Доставлено"] / pivot_table["Отправлено"])
pivot_table["Open rate"] = to_percent(pivot_table["Открытия"] / pivot_table["Доставлено"])
pivot_table["CTOR"] = to_percent(pivot_table["Клики"] / pivot_table["Открытия"])
pivot_table["UR"] = to_percent(pivot_table["Отписки"] / pivot_table["Доставлено"])

pivot_table["Sale CR 2"] = to_percent(pivot_table["Воронка продаж. Шаг 2"] / pivot_table["Воронка продаж. Шаг 1"])
pivot_table["Sale CR 3"] = to_percent(pivot_table["Воронка продаж. Шаг 3"] / pivot_table["Воронка продаж. Шаг 2"])

# pivot_table = pivot_table.drop(cols2digit, axis=1)

pivot_table.to_excel("./pivot_table.xlsx")
pivot_table.to_csv("./pivot_table.csv")

pivot_table

,Отправлено,Доставлено,Открытия,Клики,Отписки,Воронка продаж. Шаг 1,Воронка продаж. Шаг 2,Воронка продаж. Шаг 3,Deliver rate,Open rate,CTOR,UR,Sale CR 2,Sale CR 3
Дата,,,,,,,,,,,,,,
2021-04-15,688.0,654.0,94.0,3.0,4.0,659,551,435,0.9506,0.1437,0.0319,0.0061,0.8361,0.7895
2021-04-21,627.0,596.0,83.0,6.0,4.0,2428,1971,1479,0.9506,0.1393,0.0723,0.0067,0.8118,0.7504
2021-04-22,1.0,1.0,285.0,34.0,12.0,17664,17310,12637,1.0000,285.0000,0.1193,12.0000,0.9800,0.7300
2021-04-23,2.0,2.0,363.0,32.0,15.0,11634,10121,7996,1.0000,181.5000,0.0882,7.5000,0.8700,0.7900
2021-04-30,724.0,688.0,99.0,8.0,4.0,2677,2275,1888,0.9503,0.1439,0.0808,0.0058,0.8498,0.8299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-04-26,552.0,524.0,235.0,16.0,13.0,4090,3509,2791,0.9493,0.4485,0.0681,0.0248,0.8579,0.7954
2022-05-05,1.0,1.0,150.0,10.0,7.0,3852,3127,2283,1.0000,150.0000,0.0667,7.0000,0.8118,0.7301
2022-05-12,2.0,2.0,316.0,37.0,15.0,12202,11958,9447,1.0000,158.0000,0.1171,7.5000,0.9800,0.7900


# Задание №2

Построй визуализацию для набора данных из задания №1 (инструменты: DataLens или Looker Data Studio). Элементы, которые необходимо отразить в дашборде:
Динамика ключевых метрик для оценки эффективности рассылок 
Визуализируй воронку продаж

In [5]:
pivot_table[(pivot_table.index >= "2022-01-10") & (pivot_table.index < "2022-01-20")]

,Отправлено,Доставлено,Открытия,Клики,Отписки,Воронка продаж. Шаг 1,Воронка продаж. Шаг 2,Воронка продаж. Шаг 3,Deliver rate,Open rate,CTOR,UR,Sale CR 2,Sale CR 3
Дата,,,,,,,,,,,,,,
2022-01-10,1937.0,1901.0,1024.0,101.0,370.0,39136,35075,28334,0.9814,0.5387,0.0986,0.1946,0.8962,0.8078
2022-01-13,11.0,11.0,1651.0,143.0,768.0,55632,48752,38513,1.0000,150.0909,0.0866,69.8182,0.8763,0.7900
2022-01-14,967.0,952.0,771.0,62.0,656.0,24354,22341,17013,0.9845,0.8099,0.0804,0.6891,0.9173,0.7615
2022-01-17,2.0,2.0,480.0,42.0,17.0,19254,16479,13018,1.0000,240.0000,0.0875,8.5000,0.8559,0.7900
2022-01-19,881.0,864.0,807.0,54.0,7.0,15632,13090,10150,0.9807,0.9340,0.0669,0.0081,0.8374,0.7754


[Дашборд - ключевые метрики](https://datalens.ru/voa5xz5rk5fqf-klyuchevye-metriki)

# Задание №3

На основе уже знакомого тебе тестового набора данных из первого задания посчитай следующие метрики в тетрадке Jupyter Notebook:
- Delivery rate
- Open rate
- Click to Open rate
- Unsubscribe rate
- Выяви лучшую тему

In [8]:
deliver_rate = pivot_table[(pivot_table["Отправлено"] >= pivot_table["Доставлено"])]
deliver_rate = deliver_rate["Доставлено"].sum() / deliver_rate["Отправлено"].sum()

open_rate = pivot_table[(pivot_table["Доставлено"] >= pivot_table["Открытия"])]
open_rate = open_rate["Открытия"].sum() / open_rate["Доставлено"].sum()

ctor = pivot_table[(pivot_table["Открытия"] >= pivot_table["Клики"])]
ctor = ctor["Клики"].sum() / ctor["Открытия"].sum()

ur = pivot_table[(pivot_table["Доставлено"] >= pivot_table["Отписки"])]
ur = ur["Отписки"].sum() / ur["Доставлено"].sum()

subjects = df.groupby(["Тема письма "], as_index=False).agg({title: "sum" for title in ["Доставлено", "Открытия"]})
subjects = subjects[subjects["Доставлено"] >= subjects["Открытия"]]
subjects["Open rate"] = subjects["Открытия"] / subjects["Доставлено"]
subjects = subjects.sort_values(["Open rate"], ascending=False)

best_subject, delivered, opened, open_rate = subjects[:1].values[0]

print(f"""
Deliver rate = {deliver_rate:.2%}
Open rate = {open_rate:.2%}
CTOR = {ctor:.2%}
UR = {ur:.2%}

Лучшая тема - "{best_subject}", Open rate = {open_rate:.2%}
""")


Deliver rate = 97.53%
Open rate = 19.97%
CTOR = 7.85%
UR = 7.07%

Лучшая тема - "Тема письма 1", Open rate = 19.97%

